In [8]:
from pathlib import Path
import json
from ultralytics import YOLO
import random
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple
import time
import shutil
import cv2

from annotation_methods.budget_splits import make_nested_splits, Params
from annotation_methods.yolo_helpers import convert_all_inst_splits_to_yolo, write_yolo_dataset_yaml
from annotation_methods.io_utils import write_coco_output

In [9]:
# ---- Reusable constants ----
DATA_ROOT = Path("../../Data")
DATA_YAML_NAME = "dataset.yaml"
STATS_JSON_NAME = "stats.json"
RUNS_DIR = Path("runs")
MODEL_WEIGHTS = "yolo11n.pt"
INSTANCE_BUDGETS = (250, 500, 1000)

RESULTS_PATH = Path("../../Results/Experiment_1")

DATASETS = ["apples", "tomatoes"]
DATASET_DICT = {
    "apples": ["good apple", "bad apple"],
    "tomatoes": ["tomato"],
}

## make the splits and yolo yaml file

In [10]:
for dataset in DATASETS:
    make_nested_splits(
        train_json=DATA_ROOT / dataset / "annotations" / "instances_train.json",
        out_root=DATA_ROOT / dataset / "yolo_splits",
        params=Params(INSTANCE_BUDGETS, val_frac=0.2, seed=42),
    )

Wrote splits to: /home/warredv/Thesis_WDV/Data/apples/yolo_splits
Wrote splits to: /home/warredv/Thesis_WDV/Data/tomatoes/yolo_splits


In [11]:
for dataset in DATASETS:
    all_results = convert_all_inst_splits_to_yolo(
        inst_values=INSTANCE_BUDGETS,
        coco_json=DATA_ROOT / dataset / "annotations" / "instances_train.json",
        dataset_root=DATA_ROOT / dataset,
        splits_root=DATA_ROOT / dataset / "yolo_splits",
    )
    for r in all_results:
        print(f"{dataset} inst{r['inst']} TRAIN:", r["train"])
        print(f"{dataset} inst{r['inst']} VAL  :", r["val"])

apples inst250 TRAIN: {'processed_images': 16, 'labels_written': 208, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst250 VAL  : {'processed_images': 3, 'labels_written': 52, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst500 TRAIN: {'processed_images': 29, 'labels_written': 398, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst500 VAL  : {'processed_images': 8, 'labels_written': 108, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst1000 TRAIN: {'processed_images': 57, 'labels_written': 792, 'missing_images_on_disk': 0, 'missing_manifest_entries_in_coco': 0, 'nc': 2, 'class_names': ['GoodApple', 'BadApple']}
apples inst1000 VAL  : {'processed_images': 16, 'labels_w

In [12]:
for dataset in DATASETS:
    for inst in INSTANCE_BUDGETS:
        split_dir = DATA_ROOT / dataset / "yolo_splits" / f"inst{inst}"
        write_yolo_dataset_yaml(split_dir)

## train models

In [13]:
def train_yolo_model(dataset: str, split: str, ):
    """
    Train a YOLO model for a given dataset + split and
    save timing stats to training_info.json and store the 
    model in the appropriate runs/dataset folder.
    """

    # ---- Paths derived from inputs ----
    split_path = Path(split)
    data_dir = DATA_ROOT / dataset / split_path
    data_yaml_path = data_dir / DATA_YAML_NAME
    stats_json_path = data_dir / STATS_JSON_NAME

    project_dir = RUNS_DIR / dataset
    num_instances = split_path.name.replace("inst", "")
    model_name = f"yolo11n_inst{num_instances}"

    # ---- Load model ----
    model = YOLO(MODEL_WEIGHTS)

    # ---- Train and time ----
    start = time.perf_counter()
    train_results = model.train(
        data=str(data_yaml_path),
        epochs=100,
        imgsz=640,
        device="cuda",
        project=str(project_dir),
        name=model_name,
    )
    end = time.perf_counter()
    machine_training_time_s = end - start

    # ---- Read stats.json if available ----
    if stats_json_path.exists():
        with stats_json_path.open("r") as f:
            stats = json.load(f)
        num_initial_bbox = stats.get("budget_instances_actual_pool", 0)
    else:
        num_initial_bbox = 0

    # ---- Save training info ----
    output_path = project_dir / model_name / "training_info.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    training_info = {
        "machine_training_time_s": machine_training_time_s,
        "num_initial_bbox": num_initial_bbox,
    }

    with output_path.open("w") as f:
        json.dump(training_info, f, indent=2)

    # ---- Return useful outputs for further processing ----
    return {
        "model_name": model_name,
        "project_dir": project_dir,
        "train_results": train_results,
        "machine_training_time_s": machine_training_time_s,
        "num_initial_bbox": num_initial_bbox,
        "training_info_path": output_path,
    }

In [15]:
result = train_yolo_model(dataset="apples", split="yolo_splits/inst500")
print(result["model_name"], result["machine_training_time_s"], result["num_initial_bbox"])

Ultralytics 8.3.241 🚀 Python-3.11.14 torch-2.9.1 CUDA:0 (NVIDIA A10G, 22588MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../Data/apples/yolo_splits/inst500/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11n_inst500, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspect

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8404.7±1609.7 MB/s, size: 3473.0 KB)
train: Scanning /home/warredv/Thesis_WDV/Data/apples/yolo_splits/inst500/labels/train.cache... 29 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 29/29 75.7Kit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2481.0±2156.1 MB/s, size: 3463.6 KB)
val: Scanning /home/warredv/Thesis_WDV/Data/apples/yolo_splits/inst500/labels/val.cache... 8 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8/8 15.2Kit/s 0.0ss
Plotting labels to /home/warredv/Thesis_WDV/Models/Yolo/runs/apples/yolo11n_inst500/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.001667, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/warr

In [16]:
def run_inference(dataset: str, model_name: str, outdir_base: Path = Path("yolo_outputs")):
    """
    Run a YOLO model on the test split for a given dataset and
    save timing + bbox stats in outdir_base/experiment_name with:
     - YOLO bbox information in a subfolder 'labels' with seperate txt files
     - timing information in training_info.json.
    """
    experiment_name = f'{dataset}_test_{model_name}'

    model_path = RUNS_DIR / dataset / model_name / "weights" / "best.pt"
    model = YOLO(str(model_path))

    test_images_path = DATA_ROOT / dataset / "images" / "test"
    outdir_base = outdir_base 

    training_info_path = RUNS_DIR / dataset / model_name / "training_info.json"

    # --- prepare output directory ---
    exp_outdir = outdir_base / experiment_name
    exp_outdir.mkdir(parents=True, exist_ok=True)

    # --- clean previous prediction files ---
    labels_dir = exp_outdir / "labels"
    if labels_dir.exists():
        shutil.rmtree(labels_dir)   # delete old txt files
    labels_dir.mkdir(parents=True, exist_ok=True)

    # --- run inference ---
    start = time.perf_counter()
    model.predict(
        source=str(test_images_path),
        save=False,
        save_txt=True,
        save_conf=True,
        project=str(outdir_base),
        name=experiment_name,  # will now point to a clean folder
        exist_ok=True,
    )
    total_inference_time_s = time.perf_counter() - start

    # --- write timing info ---
    with training_info_path.open("r") as f:
        training_info = json.load(f)

    timing_info = {
        "total_inference_time_s": total_inference_time_s,
        "machine_training_time_s": training_info["machine_training_time_s"],
        "num_initial_bbox": training_info["num_initial_bbox"],
    }

    with (exp_outdir / "timing_info.json").open("w") as f:
        json.dump(timing_info, f, indent=2)

In [17]:
#run for every dataset, for every model (run the first model twice as warmup)
dataset = "apples"
model_name = "yolo11n_inst500"
run_inference(dataset=dataset, model_name=model_name)


image 1/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0298.png: 544x640 12 GoodApples, 8 BadApples, 69.9ms
image 2/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0302.png: 544x640 12 GoodApples, 8 BadApples, 7.4ms
image 3/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0307.png: 544x640 12 GoodApples, 8 BadApples, 7.3ms
image 4/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0326.png: 544x640 12 GoodApples, 7.4ms
image 5/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0335.png: 544x640 12 GoodApples, 7.3ms
image 6/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0337.png: 544x640 12 GoodApples, 7.3ms


image 7/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0345.png: 544x640 12 GoodApples, 8.1ms
image 8/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0359.png: 544x640 12 GoodApples, 7.3ms
image 9/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0360.png: 544x640 12 GoodApples, 7.4ms
image 10/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0362.png: 544x640 12 GoodApples, 7.3ms
image 11/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0364.png: 544x640 12 GoodApples, 7.3ms
image 12/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0365.png: 544x640 12 GoodApples, 1 BadApple, 7.5ms
image 13/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0371.png: 544x640 12 GoodApples, 1 BadApple, 7.3ms
image 14/31 /home/warredv/Thesis_WDV/Models/Yolo/../../Data/apples/images/test/IMG_0372.png: 544x640 12 GoodApples, 1 BadAppl

In [18]:
def yolo_pred_txt_to_coco_results(
    dataset,
    model_name,
    categories_list,
    output_path = RESULTS_PATH,
    is_xywh_normalized=True,
    has_conf=True,
):
    """
    Convert YOLO prediction .txt files into a COCO-format JSON file written via
    `write_coco_output()`.

    YOLO txt format assumed: cls xc yc w h [conf]
    """

    experiment_name = f'{dataset}_test_{model_name}'
    test_images_path = DATA_ROOT / dataset / "images" / "test"

    yolo_experiment_outputs = Path("yolo_outputs") / experiment_name
    yolo_labels_path = yolo_experiment_outputs / "labels"
    yolo_timing_info_path = yolo_experiment_outputs / "timing_info.json"

    # Index images by stem (filename without extension)
    img_index = {}
    for p in test_images_path.rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}:
            img_index[p.stem] = p

    images = []
    annotations = []
    ann_id = 1
    img_id = 1
    num_pred_boxes = 0

    for txt in sorted(yolo_labels_path.glob("*.txt")):
        stem = txt.stem
        img_path = img_index.get(stem)
        if img_path is None:
            continue

        im = cv2.imread(str(img_path))
        if im is None:
            continue
        h, w = im.shape[:2]

        images.append(
            {
                "id": img_id,
                "file_name": f"images/test/{img_path.name}",
                "width": int(w),
                "height": int(h),
            }
        )

        with txt.open("r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue

                cls = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:5])
                conf = float(parts[5]) if (has_conf and len(parts) >= 6) else 1.0

                if is_xywh_normalized:
                    xc *= w
                    yc *= h
                    bw *= w
                    bh *= h

                x = xc - bw / 2.0
                y = yc - bh / 2.0

                coco_cat = cls  # Assuming 0-based indexing

                annotations.append(
                    {
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": int(coco_cat),
                        "bbox": [float(x), float(y), float(bw), float(bh)],
                        "score": float(conf),
                    }
                )
                ann_id += 1
                num_pred_boxes += 1

        img_id += 1


    # Read timing info
    with yolo_timing_info_path.open("r") as f:
        timing_info = json.load(f)

    machine_training_time_s= float(timing_info.get("machine_training_time_s", 0.0))
    total_inference_time_s= float(timing_info.get("total_inference_time_s", 0.0))
    num_initial_bbox = int(timing_info.get("num_initial_bbox", 0))

    return write_coco_output(
        images_folder=str(test_images_path),
        model_name=model_name,
        categories_list=categories_list,
        images=images,
        annotations=annotations,
        num_images=len(images),
        num_initial_bbox=num_initial_bbox,
        num_pred_boxes=num_pred_boxes,
        machine_training_time_s=float(machine_training_time_s),
        total_inference_time_s=float(total_inference_time_s),
        output_path=str(output_path),
    )


In [19]:
dataset="apples"
model_name="yolo11n_inst500"
categories_list = DATASET_DICT.get("apples") #just needs to be the same length as num classes

yolo_pred_txt_to_coco_results(
    dataset=dataset,
    model_name=model_name,
    categories_list=categories_list,
    output_path=RESULTS_PATH,          # directory, no filename
    is_xywh_normalized=True,
    has_conf=True,
)

'../../Results/Experiment_1/apples_test_yolo11n_inst500_predictions.json'